# CNN sur Fashion-MNIST
Même démarche que pour le MLP : on calibre d'abord (largeur × epochs, sans dropout vs dropout classique) pour trouver un régime de sous-apprentissage, puis on lance la grille des durées d'early dropout. À lancer sur Colab avec le GPU.

In [ ]:
# Colab : monte le Drive et se place dans le repo
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout
!git pull

In [ ]:
%load_ext autoreload
%autoreload 2

import os, glob, time
import pandas as pd
import torch
from torchvision import datasets

from config import BATCH, P, EARLY_GRID
from models import cnn
from train import entrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
tr = datasets.FashionMNIST("./data", train=True, download=True)
te = datasets.FashionMNIST("./data", train=False, download=True)
Xtr = tr.data.float().unsqueeze(1) / 255      # (N, 1, 28, 28) : on garde la forme d'image pour le CNN
Xte = te.data.float().unsqueeze(1) / 255
ytr, yte = tr.targets, te.targets

# même split de validation que pour le MLP
perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(0))
Xva, yva = Xtr[perm[:10000]], ytr[perm[:10000]]
Xtr, ytr = Xtr[perm[10000:]], ytr[perm[10000:]]

# normalisation avec les stats du train uniquement
mu, sd = Xtr.mean(), Xtr.std()
Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

data = dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte)
data = {k: v.to(device) for k, v in data.items()}       # tout sur le GPU
print(Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
CADRE = "cnn_fmnist"
OUT = "resultats"

# combien de temps prend une epoch ? (pour dimensionner les grilles)
torch.manual_seed(0)
model, opt = cnn(1, 10, largeur=8)
model.to(device)
t0 = time.time()
entrainer(model, opt, data, epochs=1, t_early=0, p=P, batch=BATCH)
print(f"{time.time() - t0:.1f} s par epoch")

## Calibration : trouver un régime de sous-apprentissage
On compare sans dropout et dropout classique. delta = acc val (dropout classique) - acc val (sans dropout) : si delta < 0 le modèle sous-apprend, c'est ce qu'on cherche.

In [ ]:
CALIB_LARGEURS = [4, 8, 16]
CALIB_EPOCHS = [10, 20]
CALIB_GRID = [0, 1]          # 0 = sans dropout, 1 = dropout classique
CALIB_SEEDS = [0, 1]

for L in CALIB_LARGEURS:
    for E in CALIB_EPOCHS:
        for frac in CALIB_GRID:
            for seed in CALIB_SEEDS:
                fichier = f"{OUT}/{CADRE}_w{L}_e{E}_frac{frac}_s{seed}.csv"
                if os.path.exists(fichier):
                    continue
                torch.manual_seed(seed)
                model, opt = cnn(1, 10, largeur=L)
                model.to(device)
                hist = entrainer(model, opt, data, E, t_early=round(frac * E), p=P, batch=BATCH)
                hist.assign(cadre=CADRE, width=L, epochs=E, frac=frac, seed=seed).to_csv(fichier, index=False)

In [ ]:
cal = pd.concat(pd.read_csv(f) for f in glob.glob(f"{OUT}/{CADRE}_w*_e*_frac*_s*.csv"))
cal = cal[cal.width.isin(CALIB_LARGEURS) & cal.epochs.isin(CALIB_EPOCHS) & cal.frac.isin(CALIB_GRID)]

# valeur finale = moyenne des 5 dernières epochs de chaque run
fin = (cal[cal.epoch >= cal.epochs - 5]
       .groupby(["width", "epochs", "frac", "seed"])[["train_loss", "val_loss", "val_acc"]].mean()
       .reset_index())

t = fin.pivot_table(index=["width", "epochs", "seed"], columns="frac", values="val_acc")
t["delta"] = (t[1] - t[0]) * 100                         # dropout classique - sans dropout, en points
tab = t.groupby(["width", "epochs"]).agg(
    acc_sans_dropout=(0, "mean"),
    delta_moy=("delta", "mean"),
    delta_min=("delta", "min"),
    delta_max=("delta", "max"),
)
tab["sous_apprentissage"] = tab.delta_max <= -0.5         # toutes les seeds perdent au moins 0.5 pt
tab.round(3)

On retient une case où `sous_apprentissage` est True, avec une accuracy correcte. Ensuite on lance la grille complète comme pour le MLP.

In [ ]:
LARGEUR = 8      # à remplacer par la largeur retenue
EPOCHS = 20      # à remplacer par le budget retenu
SEEDS = [0, 1, 2, 3, 4]

for frac in EARLY_GRID:
    for seed in SEEDS:
        fichier = f"{OUT}/{CADRE}_w{LARGEUR}_e{EPOCHS}_frac{frac}_s{seed}.csv"
        if os.path.exists(fichier):
            continue
        torch.manual_seed(seed)
        model, opt = cnn(1, 10, largeur=LARGEUR)
        model.to(device)
        hist = entrainer(model, opt, data, EPOCHS, t_early=round(frac * EPOCHS), p=P, batch=BATCH)
        hist.assign(cadre=CADRE, width=LARGEUR, epochs=EPOCHS, frac=frac, seed=seed).to_csv(fichier, index=False)

In [ ]:
# gain par rapport à "sans dropout", seed par seed, avec IC à 95 % (Student, 5 seeds)
metriques = ["train_loss", "val_loss", "val_acc", "test_acc"]
fin = {}
for frac in EARLY_GRID:
    fin[frac] = pd.DataFrame(
        [pd.read_csv(f"{OUT}/{CADRE}_w{LARGEUR}_e{EPOCHS}_frac{frac}_s{s}.csv").tail(5)[metriques].mean() for s in SEEDS],
        index=SEEDS)

res = {}
for frac in EARLY_GRID:
    if frac == 0:
        continue
    d = fin[frac] - fin[0]
    d[["val_acc", "test_acc"]] *= 100                     # accuracy en points
    ic = 2.776 * d.std() / len(SEEDS) ** 0.5
    res[frac] = pd.concat([d.mean(), ic.add_prefix("ic_")])
pd.DataFrame(res).T.round(3)